# 08 — Segmentation, Risk Score & Final Knowledge Layer (Member 3)

**Phase 3 (ML) + Phase 4 (final).** Adds `cluster_label`, `cluster_distance`, and `risk_priority_score` to Member 2's KPI table and writes the final `data/processed/knowledge_layer.csv` that Member 4's chatbot retrieves from.

Logic lives in `src/segmentation.py` (pure functions); this notebook is a thin orchestrator. See `reports/model_metrics.md` for the written analysis.

In [1]:
import sys
from pathlib import Path
# Resolve repo root whether run from notebooks/ or repo root
_here = Path.cwd()
ROOT = _here if (_here / 'src').exists() else _here.parent
sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np, joblib
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.data_io import PROCESSED_DIR
print('repo root:', ROOT)

repo root: /Users/bobbypakenham/KPMG_Airbnb_Capstone


## Load the input KPI table (560 × 27)

In [2]:
from src import segmentation as seg
kpis = pd.read_csv(PROCESSED_DIR / 'neighbourhood_kpis.csv')
print(kpis.shape)
kpis[['city','geo_key'] + seg.CLUSTER_FEATURES].head()

(560, 27)


,city,geo_key,str_density,entire_home_share,commercial_host_share,multi_listing_host_share,avg_occupancy,breach_rate_90
0,barcelona,Can Baró,7,0.5714,0.1429,0.4286,0.1951,0.5000
1,barcelona,Diagonal Mar i el Front Marítim del Poblenou,19,0.6842,0.1053,0.4211,0.1050,0.3077
2,barcelona,Gothic Quarter,160,0.4438,0.2125,0.4688,0.1091,0.2535
3,barcelona,Horta,1,0.0000,0.0000,0.0000,0.0000,NaN
4,barcelona,Hostafrancs,32,0.4375,0.2812,0.4688,0.1913,0.2857


## 1. KMeans clustering (k=3)

Features are StandardScaler-transformed (a count + several shares share the matrix). Labels are assigned by centroid **pressure rank**, so `saturated`/`emerging`/`low_impact` are stable across reruns regardless of KMeans' arbitrary cluster ids.

In [3]:
clustered, meta = seg.compute_clusters(kpis)
print('silhouette (k=3):', meta['silhouette'])
meta['centroids'][['cluster_label','n_members'] + seg.CLUSTER_FEATURES + ['pressure_score']].round(3)

silhouette (k=3): 0.2715


,cluster_label,n_members,str_density,entire_home_share,commercial_host_share,multi_listing_host_share,avg_occupancy,breach_rate_90,pressure_score
0,saturated,189,46.069,0.719,0.194,0.467,0.118,0.243,0.790
1,low_impact,293,9.218,0.469,0.019,0.118,0.052,0.084,0.011
2,emerging,78,6.987,0.642,0.032,0.217,0.242,0.614,0.521


### Cross-check vs Member 2's rule-based tiers
All tier-1 areas should land in `saturated`.

In [4]:
kl_preview = seg.assemble_knowledge_layer(kpis, clustered)
seg.tier_cluster_crosstab(kl_preview)

cluster_label,emerging,low_impact,saturated
tier_concentration_price,,,
tier_1,0,0,37
tier_2,9,26,73
tier_3,69,267,79


## 2. Risk Priority Score (0–100)

Per-city min-max weighted composite, shrunk by `density/(density+5)` so single-listing areas can't top the ranking on n=1 noise, rescaled so each city's worst = 100.

In [5]:
kl = seg.assemble_knowledge_layer(kpis, clustered)
for city, g in kl.groupby('city'):
    print(f'\n{city} — top 8 by risk_priority_score')
    print(g.nlargest(8,'risk_priority_score')[['geo_key','cluster_label','risk_priority_score','str_density','entire_home_share','breach_rate_90','tier_concentration_price']].to_string(index=False))


barcelona — top 8 by risk_priority_score
                        geo_key cluster_label  risk_priority_score  str_density  entire_home_share  breach_rate_90 tier_concentration_price
         la Dreta de l'Eixample     saturated               100.00          295             0.6814          0.4726                   tier_1
                   el Poble-sec     saturated                91.06          118             0.7034          0.5542                   tier_1
             la Sagrada Família     saturated                88.09          144             0.7361          0.5000                   tier_1
l'Antiga Esquerra de l'Eixample     saturated                86.19          148             0.6351          0.4574                   tier_2
 la Nova Esquerra de l'Eixample     saturated                83.68          114             0.5263          0.5167                   tier_2
                    Sant Antoni     saturated                78.76           79             0.6456          0.5098    

## 3. Write the final knowledge layer

In [6]:
out = PROCESSED_DIR / 'knowledge_layer.csv'
kl.to_csv(out, index=False)
print('wrote', out, kl.shape)
print('new M3 columns:', [c for c in ['cluster_label','cluster_distance','risk_priority_score'] if c in kl.columns])

wrote /Users/bobbypakenham/KPMG_Airbnb_Capstone/data/processed/knowledge_layer.csv (560, 35)
new M3 columns: ['cluster_label', 'cluster_distance', 'risk_priority_score']


## 4. Figures

In [7]:
import subprocess, sys as _s
# Figures are generated by the same logic captured in src; regenerate via the helper if desired.
fig, ax = plt.subplots(figsize=(7,4))
COL={'saturated':'#c0392b','emerging':'#e67e22','low_impact':'#27ae60'}
pd.crosstab(kl['city'],kl['cluster_label'])[['saturated','emerging','low_impact']].plot(kind='bar',ax=ax,color=[COL['saturated'],COL['emerging'],COL['low_impact']])
ax.set_title('Cluster composition by city'); plt.xticks(rotation=0); fig.tight_layout()
Path('../reports/figures/clustering').mkdir(parents=True, exist_ok=True)
fig.savefig(ROOT/'reports'/'figures'/'clustering'/'cluster_by_city.png', dpi=130); print('saved cluster_by_city.png')

saved cluster_by_city.png
